# V3 / 02C — Three simple Musanga models and held-out fractional-cover validation

M1: partial crown U-Net; M2: same partial crowns plus exhaustive DEVELOPMENT circles, uncertain contradictions ignored; M3: M1 with circle-level probability calibration. Seven (~30%) circles remain untouched for evaluation. Primary results are observed vs predicted circle fraction, R², MAE, RMSE, bias, regression slope and intercept. No full-mosaic inference.


## 01 — Inputs and frozen design


In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas matplotlib scikit-learn
from pathlib import Path
import json, warnings, math, random, time
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score, roc_auc_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OUT=ROOT/'Outputs/V3/02_simple';OUT.mkdir(parents=True,exist_ok=True)
REF=ROOT/'Outputs/V3/01_reference_data'
A=json.loads((ROOT/'Outputs/V3/00_input_audit/00_input_manifest.json').read_text())
mosaics={m['file']:m for m in A['mosaics']}
splits=pd.read_csv(REF/'01_cv_window_manifest.csv')
links=pd.read_csv(REF/'01_circle_spatial_folds.csv')
repairs=pd.read_csv(REF/'01_geometry_repairs.csv')
FOLD=0;SEED=20261016;CROP=256;EPOCHS=25;BATCH=8
MAX_PARTIAL_PER_MOSAIC=250
MAX_EXHAUSTIVE_PER_SITE=260
HOLDOUT_COUNTS={'Baego':1,'MOS':2,'Yangambi':2,'Yoko':2}
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > GPU for the pilot.'
device='cuda';torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
print('GPU',torch.cuda.get_device_name(0),'| fold',FOLD)
print('Train/val candidates:',splits[splits.cv_fold==FOLD].split.value_counts().to_dict())
print('Files:',ROOT,'| output:',OUT)





## 02 — Label geometry and a fixed geographically separated 30% holdout


In [ ]:
def pieces(g):
    if g.geom_type in ('Polygon','MultiPolygon'):return [g]
    if hasattr(g,'geoms'):return [p for q in g.geoms for p in pieces(q)]
    return []
def polygons(path):
    src=gpd.read_file(path);inds=[];shapes=[]
    for i,g in enumerate(src.geometry):
        if g is None or g.is_empty:continue
        if not g.is_valid:
            try:
                fixed=unary_union(pieces(make_valid(g)))
                change=abs(fixed.area-g.area)/max(fixed.area,g.area,1e-12)
                if change>.05 or not fixed.is_valid:continue
                g=fixed
            except Exception:continue
        inds.append(i);shapes.append(g)
    x=src.iloc[inds].copy().reset_index(drop=True)
    x.geometry=shapes
    x['original_row']=inds
    return x
roi=polygons(A['roi'])
circles=polygons(A['exhaustive_annotations']['complete_roi_boundaries'])
positive=polygons(A['exhaustive_annotations']['positive_polygons'])
circle_ids=set(links.circle_id.astype(int))
circles=circles[circles.original_row.isin(circle_ids)].copy()
tree={}
for site,path in A['shared_annotations'].items():
    t=polygons(path)
    cls=pd.to_numeric(t.Class_ID,errors='coerce')
    t['class01']=np.where(cls.eq(1),1,np.where(cls.notna(),0,-1))
    tree[site]=t
geo={}
for name,m in mosaics.items():
    with rasterio.open(m['path']) as src:
        c=circles.to_crs(src.crs).copy()
        c['fold']=c.original_row.map(links.set_index('circle_id').fold)
        c['owner']=c.original_row.map(links.set_index('circle_id').mosaic)
        geo[name]=dict(site=m['site'],path=m['path'],transform=src.transform,
            crs=src.crs,roi=roi.to_crs(src.crs),
            circles=c,positive=positive.to_crs(src.crs),trees=tree[m['site']].to_crs(src.crs))



## 03 — U-Net architecture and loss (same as Notebook 02B)


In [ ]:
class TinyUNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        def block(a,b):
            return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True),
                nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(inplace=True))
        self.e1=block(3,base);self.e2=block(base,base*2)
        self.bridge=block(base*2,base*4)
        self.d2=block(base*4+base*2,base*2)
        self.d1=block(base*2+base,base)
        self.final=nn.Conv2d(base,1,1)
    def forward(self,x):
        e1=self.e1(x);e2=self.e2(F.max_pool2d(e1,2))
        b=self.bridge(F.max_pool2d(e2,2))
        x=self.d2(torch.cat([F.interpolate(b,size=e2.shape[-2:],mode='bilinear',align_corners=False),e2],1))
        x=self.d1(torch.cat([F.interpolate(x,size=e1.shape[-2:],mode='bilinear',align_corners=False),e1],1))
        return self.final(x)[:,0]
class CropData(Dataset):
    def __init__(self,rows,augment=False):self.rows=rows;self.augment=augment
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        d=self.rows[i];image=d['rgb'];y=d['y'];w=d['w']
        if self.augment:
            rng=np.random.default_rng(SEED+i+int(torch.randint(0,1000000,(1,)).item()))
            k=int(rng.integers(4));image=np.rot90(image,k);y=np.rot90(y,k);w=np.rot90(w,k)
            if rng.random()<.5:image=image[:,::-1];y=y[:,::-1];w=w[:,::-1]
        x=torch.tensor(np.ascontiguousarray(image.transpose(2,0,1)),dtype=torch.float32)
        return x,torch.tensor(np.ascontiguousarray(y)),torch.tensor(np.ascontiguousarray(w))
def masked_loss(logits,y,w):
    loss=F.binary_cross_entropy_with_logits(logits,y,reduction='none')
    return (loss*w).sum()/w.sum().clamp_min(1.)
print('Tiny U-Net ready (native 256×256 RGB crops).')




## 04 — A single 15/7 split, frozen before fitting any model or calibration

Choose one or two geographically separated circles per forest for holdout. The existing Notebook 01 fold IDs guide selection; save these IDs and never change them after viewing scores. All partial training RGB chips intersecting holdout circles +15m are excluded.


In [ ]:
from scipy.spatial.distance import cdist
from rasterio.windows import Window
from rasterio.features import rasterize
CIRCLE_LINK=links.set_index('circle_id')
chosen=[]
for site,n_hold in HOLDOUT_COUNTS.items():
    block=links[links.site==site].copy()
    # Farthest-point holdout candidates while retaining some spatial diversity in development.
    centers=[]
    for cid in block.circle_id:
        g=circles[circles.original_row==int(cid)].geometry.iloc[0]
        centers.append((g.centroid.x,g.centroid.y))
    coords=np.array(centers)
    first=int(np.argmax(np.linalg.norm(coords-coords.mean(axis=0),axis=1)))
    pick=[first]
    while len(pick)<n_hold:
        distances=cdist(coords,coords[pick]).min(axis=1)
        distances[pick]=-1
        pick.append(int(distances.argmax()))
    chosen.extend(block.circle_id.iloc[pick].astype(int).tolist())
HOLDOUT=set(chosen)
DEV=set(links.circle_id.astype(int))-HOLDOUT
assert len(HOLDOUT)==7 and len(DEV)==15
split_df=links[['circle_id','site','mosaic','cover_pct']].copy()
split_df['split']=split_df.circle_id.map(lambda x:'holdout' if x in HOLDOUT else 'development')
display(split_df.sort_values(['site','split','circle_id']))
split_df.to_csv(OUT/'02C_frozen_circle_split.csv',index=False)
# Geographic exclusion in every mosaic, including across MOS mosaic boundaries.
BLOCKED={}
for name,d in geo.items():
    this=circles.to_crs(d['crs'])
    site_ids=set(links[(links.site==d['site'])&(links.circle_id.isin(HOLDOUT))].circle_id.astype(int))
    BLOCKED[name]=[g.buffer(15.) for g in this[this.original_row.isin(site_ids)].geometry]
print('Development circles:',sorted(DEV),'| independent holdout:',sorted(HOLDOUT))


## 05 — On-demand labels, RGB validity and reproducible training chips

M1 trains only on partial crowns **outside every exhaustive circle + 15m**. Thus development-circle labels are independent of M1 training and may be used for M3 calibration. M2 uses the SAME partial chips plus development-circle exhaustive supervision (conflicts ignored); the seven holdout circles are always excluded from training. All RGB is read from original TIFFs at native resolution.


In [ ]:
from rasterio.windows import Window
from torch.utils.data import Dataset,DataLoader
candidate=pd.read_csv(REF/'01_candidate_windows.csv')
all_circle_buffers={}
for name,d in geo.items():
    cs=circles.to_crs(d['crs'])
    all_circle_buffers[name]=[g.buffer(15.) for g in cs.geometry]
    d['dev_circles']=cs[cs.original_row.isin(DEV)]
    d['holdout_circles']=cs[cs.original_row.isin(HOLDOUT)]
    d['positive_local']=positive.to_crs(d['crs'])
    d['trees_local']=tree[d['site']].to_crs(d['crs')]
    d['roi_local']=roi.to_crs(d['crs'])

def burn_window(series,name,win):
    d=geo[name];bb=box(*rasterio.windows.bounds(win,d['transform']))
    gs=[g.intersection(bb) for g in series if g is not None and g.intersects(bb)]
    if not gs:return np.zeros((int(win.height),int(win.width)),bool)
    return rasterize([(g,1) for g in gs if not g.is_empty],
       out_shape=(int(win.height),int(win.width)),transform=rasterio.windows.transform(win,d['transform']),
       fill=0,dtype='uint8').astype(bool)

def rgb_for_window(name,win):
    with rasterio.open(geo[name]['path']) as ds:
        raw=ds.read([1,2,3],window=win,masked=True)
    valid=~np.any(np.ma.getmaskarray(raw),axis=0)
    a=raw.filled(0).astype('float32')
    valid &= np.any(a!=0,axis=0)&np.any(a!=255,axis=0)
    return np.clip(a/255.,0,1).transpose(1,2,0).astype('float32'),valid

def training_crop(rec,model):
    name=rec['mosaic'];d=geo[name]
    if rec.height<CROP or rec.width<CROP:return None
    rr=int(rec.row)+(int(rec.height)-CROP)//2
    cc=int(rec.col)+(int(rec.width)-CROP)//2
    win=Window(cc,rr,CROP,CROP)
    bb=box(*rasterio.windows.bounds(win,d['transform']))
    if any(bb.intersects(g) for g in BLOCKED[name]):return None
    if model=='M1' and any(bb.intersects(g) for g in all_circle_buffers[name]):return None
    if model=='M2' and rec.kind=='exhaustive' and int(rec.circle_id) not in DEV:return None
    inside=burn_window(d['roi_local'].geometry,name,win)
    t=d['trees_local']
    pos_tree=burn_window(t.loc[t.class01==1].geometry,name,win)
    neg_tree=burn_window(t.loc[t.class01==0].geometry,name,win)
    if model=='M1':
        full=np.zeros_like(inside);pos_ex=np.zeros_like(inside)
    else:
        full=burn_window(d['dev_circles'].geometry,name,win)&inside
        pos_ex=burn_window(d['positive_local'].geometry,name,win)&full
    conflict=(pos_tree&neg_tree)| (pos_ex&neg_tree) | (full&~pos_ex&pos_tree)
    target=(pos_ex|(~full&pos_tree)).astype('float32')
    weight=(full|((pos_tree|neg_tree)&~full))&inside&~conflict
    img,valid=rgb_for_window(name,win)
    weight &= valid
    if int(weight.sum())<250:return None
    return dict(image=img,target=target,weight=weight.astype('float32'),
                site=d['site'],kind=rec.kind,mosaic=name)

def make_training_cache(model):
    selected=[]
    for (site,kind),grp in candidate.groupby(['site','kind']):
        if model=='M1' and kind=='exhaustive':continue
        if model=='M2' and kind=='exhaustive':
            grp=grp[grp.circle_id.isin(DEV)]
        limit=MAX_EXHAUSTIVE_PER_SITE if kind=='exhaustive' else MAX_PARTIAL_PER_MOSAIC*(3 if site=='MOS' else 1)
        subset=grp.sample(n=min(limit,len(grp)),random_state=SEED)
        selected.extend(subset.itertuples(index=False))
    result=[]
    for i,rec in enumerate(selected):
        chip=training_crop(rec,model)
        if chip is not None:result.append(chip)
        if i and i%250==0:print(model,'loaded',i,'/',len(selected),flush=True)
    print(model,'usable training chips:',len(result),
          pd.Series([q['site'] for q in result]).value_counts().to_dict(),flush=True)
    assert len(result)>15,f'{model}: too few chips'
    return result

class CropDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        q=self.rows[i];x=q['image'].copy();y=q['target'].copy();w=q['weight'].copy()
        k=int(torch.randint(0,4,(1,)).item())
        x=np.rot90(x,k);y=np.rot90(y,k);w=np.rot90(w,k)
        if torch.rand(1).item()<.5:x=x[:,::-1];y=y[:,::-1];w=w[:,::-1]
        return (torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))),
                torch.from_numpy(np.ascontiguousarray(y)),torch.from_numpy(np.ascontiguousarray(w)))



## 06 — Train M1 and M2 (same architecture and epochs)

Save epoch losses, checkpoints and progress after every epoch. M3 does **not** train a third U-Net; it calibrates M1's probabilities. This is simpler and isolates the effect of exhaustive supervision.


In [ ]:
import time
def fit_network(label):
    ckpt=OUT/f'{label}_model.pt'
    torch.manual_seed(SEED)
    model=TinyUNet().to(device)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt,map_location=device,weights_only=True))
        print('Loaded',ckpt,flush=True)
        return model
    cache=make_training_cache(label)
    loader=DataLoader(CropDataset(cache),batch_size=BATCH,shuffle=True,num_workers=0)
    optim=torch.optim.AdamW(model.parameters(),lr=7e-4,weight_decay=1e-4)
    history=[]
    for epoch in range(EPOCHS):
        model.train();losses=[];start=time.time()
        for x,y,w in loader:
            x=x.to(device);y=y.to(device);w=w.to(device)
            optim.zero_grad(set_to_none=True)
            loss=masked_loss(model(x),y,w)
            loss.backward();optim.step()
            losses.append(float(loss.item()))
        history.append(dict(epoch=epoch+1,masked_BCE=float(np.mean(losses))))
        pd.DataFrame(history).to_csv(OUT/f'{label}_loss.csv',index=False)
        print(f'{label}: epoch {epoch+1}/{EPOCHS} BCE={np.mean(losses):.4f}, {time.time()-start:.1f}s',flush=True)
    torch.save(model.state_dict(),ckpt)
    del cache
    return model
model1=fit_network('M1')
model2=fit_network('M2')
print('M1 and M2 checkpoints ready')



## 07 — Predict full development and held-out circles, no window sampling

Inference covers the systematic 512-pixel windows from Notebook 01, clipping to **the individual circle** and the general ROI. The mean **raw probability** predicts Musanga fractional cover. We store per-circle probability histograms (256 bins) to calibrate M1 without retaining millions of pixels. Every circle is evaluated exactly once, with native RGB validity and geometric clipping.


In [ ]:
from scipy.special import expit,logit
from scipy.optimize import minimize
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
WINDOWS=pd.read_csv(REF/'01_candidate_windows.csv')
VAL_WINDOWS=WINDOWS[WINDOWS.kind=='exhaustive'].copy()
assert set(VAL_WINDOWS.circle_id.unique().astype(int))==set(links.circle_id.astype(int))
N_BINS=256
BINS=np.linspace(0,1,N_BINS+1)
BIN_CENTER=(BINS[:-1]+BINS[1:])/2

def run_all_circles(model,model_name):
    statfile=OUT/f'{model_name}_circle_raw.csv'
    histfile=OUT/f'{model_name}_histograms.npz'
    if statfile.exists() and histfile.exists():
        return pd.read_csv(statfile),dict(np.load(histfile))
    model.eval();rows=[];histograms={}
    for _,circle in split_df.iterrows():
        cid=int(circle.circle_id)
        name=circle.mosaic
        d=geo[name]
        windows=VAL_WINDOWS[(VAL_WINDOWS.circle_id==cid)&(VAL_WINDOWS.mosaic==name)]
        assert len(windows),f'No evaluation windows for circle {cid}'
        cg=circles[circles.original_row==cid].to_crs(d['crs'])
        counts=np.zeros(N_BINS,np.int64);true_counts=np.zeros(N_BINS,np.int64)
        n=0;sumy=0.;sump=0.;sum_sq=0.;tp=fp=fn=0
        rng=np.random.default_rng(SEED+cid)
        sample_truth=[];sample_probs=[]
        for rec in windows.itertuples(index=False):
            win=Window(int(rec.col),int(rec.row),int(rec.width),int(rec.height))
            mask=burn_window(cg.geometry,name,win)
            mask &= burn_window(d['roi_local'].geometry,name,win)
            rgb,valid=rgb_for_window(name,win)
            mask &= valid
            if not mask.any():continue
            reference=burn_window(d['positive_local'].geometry,name,win)
            x=torch.from_numpy(rgb.transpose(2,0,1).copy())[None].to(device)
            with torch.inference_mode():
                pred=torch.sigmoid(model(x))[0].cpu().numpy()
            y=reference[mask].astype('uint8');p=pred[mask].astype('float32')
            n+=len(y);sumy+=float(y.sum());sump+=float(p.sum())
            sum_sq+=float(np.square(p-y).sum())
            binary=p>=.5
            tp+=int(np.sum(binary&(y==1)));fp+=int(np.sum(binary&(y==0)))
            fn+=int(np.sum((~binary)&(y==1)))
            ix=np.minimum((p*N_BINS).astype(np.int32),N_BINS-1)
            counts+=np.bincount(ix,minlength=N_BINS).astype('int64')
            true_counts+=np.bincount(ix,weights=y,minlength=N_BINS).astype('int64')
            # only diagnostic AUC/AP uses pixel subsampling, with fixed per-circle cap
            take=rng.choice(len(y),min(100,len(y)),replace=False)
            sample_truth.extend(y[take].tolist());sample_probs.extend(p[take].tolist())
        assert n>0,f'No valid ROI/RGB reference pixels for circle {cid}'
        assert int(counts.sum())==n
        yt=np.asarray(sample_truth);pr=np.asarray(sample_probs)
        rows.append(dict(model=model_name,circle_id=cid,site=circle.site,split=circle.split,
                         valid_pixels=n,observed_fraction=100*sumy/n,
                         predicted_fraction=100*sump/n,
                         bias_pp=100*(sump-sumy)/n,brier=sum_sq/n,
                         f1_threshold_05=2*tp/max(1,2*tp+fp+fn),
                         auc_sample=roc_auc_score(yt,pr) if len(np.unique(yt))>1 else np.nan,
                         ap_sample=average_precision_score(yt,pr) if len(np.unique(yt))>1 else np.nan))
        histograms[str(cid)+'_count']=counts
        histograms[str(cid)+'_positive']=true_counts
        print(model_name,'circle',cid,'obs',round(100*sumy/n,1),
              'pred',round(100*sump/n,1),'valid pixels',n,flush=True)
    frame=pd.DataFrame(rows)
    assert len(frame)==22
    frame.to_csv(statfile,index=False)
    np.savez_compressed(histfile,**histograms)
    return frame,histograms

m1, hist1=run_all_circles(model1,'M1')
m2, hist2=run_all_circles(model2,'M2')
print('Full-circle results for both models saved. Calibration next.')



## 08 — M3 calibration uses only 15 development circles

Fit **two global sigmoid parameters** to reduce differences between observed and mean predicted circle fraction. Calibrator uses the **distribution of M1 pixel predictions** within each development circle, not their spatial identity. It is regularized toward the identity mapping to avoid extreme corrections from only 15 circles. No held-out circle enters optimization. Site-specific calibration is deliberately not attempted with so few circles.


In [ ]:
def calibrated_bin_probs(theta):
    a,b=np.asarray(theta,dtype=float)
    return expit(a*logit(np.clip(BIN_CENTER,1e-5,1-1e-5))+b)

dev_table=m1[m1.split=='development'].copy()
def calibration_objective(theta):
    values=calibrated_bin_probs(theta)
    errors=[]
    for row in dev_table.itertuples(index=False):
        counts=hist1[str(int(row.circle_id))+'_count']
        fraction=100*np.dot(counts,values)/max(1,counts.sum())
        errors.append(fraction-row.observed_fraction)
    # penalize large deviations from identity; circle-level, not pseudo-replicated pixels
    return float(np.mean(np.square(errors))+2.0*((theta[0]-1)**2+theta[1]**2))
fit=minimize(calibration_objective,x0=[1.0,0.0],
             bounds=[(.25,3.),(-3.,3.)],method='L-BFGS-B')
assert fit.success,fit.message
CAL_A,CAL_B=map(float,fit.x)
print('M3 calibration: sigmoid(',round(CAL_A,4),'× logit(p) +',round(CAL_B,4),')')
print('Calibration optimized on',len(dev_table),'development circles only')
(OUT/'M3_calibration.json').write_text(json.dumps(
    dict(a=CAL_A,b=CAL_B,fit_on_circle_ids=sorted(DEV),holdout_circle_ids=sorted(HOLDOUT),
         regularization=2.0,method='circle_fraction_regularized_logit_sigmoid'),indent=2))
rows=[]
cal_bin=calibrated_bin_probs([CAL_A,CAL_B])
for row in m1.itertuples(index=False):
    cid=int(row.circle_id)
    cnt=hist1[str(cid)+'_count'].astype('float64')
    pos=hist1[str(cid)+'_positive'].astype('float64')
    predicted=100*np.dot(cnt,cal_bin)/cnt.sum()
    brier=(np.dot(pos,(1-cal_bin)**2)+np.dot(cnt-pos,cal_bin**2))/cnt.sum()
    rows.append(dict(model='M3',circle_id=cid,site=row.site,split=row.split,
                     valid_pixels=row.valid_pixels,observed_fraction=row.observed_fraction,
                     predicted_fraction=predicted,bias_pp=predicted-row.observed_fraction,
                     brier=brier))
m3=pd.DataFrame(rows);m3.to_csv(OUT/'M3_circle_calibrated.csv',index=False)
print('M3 development and untouched holdout predictions ready.')

